In [26]:
# imports

using JuMP
using SCIP
import MathOptInterface as MOI
using PrettyTables
include("util.jl")

parse_scip_solution (generic function with 1 method)

In [27]:
# Файл с входными данными

input_files_folder = "input"
# input_file_name = "input_smaller.txt"
input_file_name = "input_current.txt"

"input_current.txt"

In [28]:
# Параметры модели Банистера
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7
α = 0.5

0.5

In [29]:
# Зачитывание входных данных

function read_input(file_path::String, validate::Bool)
    lines = readlines(file_path)

    lines = filter(line -> !isempty(line) && !startswith(strip(line), "#"), lines)
    lines = map(line -> split(line, "#")[1] |> strip, lines)

    idx = 1

    D = parse(Int, lines[idx]); idx += 1
    D_ = parse.(Int, split(lines[idx])); idx += 1
    D_star = parse.(Int, split(lines[idx])); idx += 1
    P = parse(Int, lines[idx]); idx += 1
        Player_names = split(lines[idx]); idx += 1
    S = parse(Int, lines[idx]); idx += 1
        Skill_names = split(lines[idx]); idx += 1
    T = parse(Int, lines[idx]); idx += 1
        Tactic_names = split(lines[idx]); idx += 1
    E = parse(Int, lines[idx]); idx += 1
        Exercise_names = split(lines[idx]); idx += 1

    a0 = parse.(Int, split(lines[idx])); idx += 1
    b0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
    c0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
    h = [parse.(Int, split(lines[idx + i - 1])) for i in 1:P]; idx += P
    w = parse.(Int, split(lines[idx])); idx += 1
    b = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
    c = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
    v = parse.(Int, split(lines[idx])); idx += 1
    V = parse.(Int, split(lines[idx])); idx += 1

    B_star = Array{Int, 3}(undef, S, P, length(D_star))
    for d in 1:length(D_star)
        for s in 1:S
            B_star[s, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

    C_star = Array{Int, 3}(undef, T, P, length(D_star))
    for d in 1:length(D_star)
        for t in 1:T
            C_star[t, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

    # input validation
    @assert maximum(D_) <= D
    @assert minimum(D_) >= 1
    @assert maximum(D_star) <= D
    @assert minimum(D_star) >= 1
    @assert length(Player_names) == P
    @assert length(Skill_names) == S
    @assert length(Tactic_names) == T
    @assert length(Exercise_names) == E
    @assert length(a0) == P
    @assert size(b0) == (S,) && size(b0[1]) == (P,)
    @assert size(c0) == (T,) && size(c0[1]) == (P,)
    @assert size(h) == (P,) && size(h[1]) == (D,)
    @assert length(w) == E
    @assert size(b) == (S,) && size(b[1]) == (E,)
    @assert size(c) == (T,) && size(c[1]) == (E,)
    @assert length(v) == E
    @assert length(V) == D
    @assert size(B_star) == (S, P, length(D_star))
    @assert size(C_star) == (T, P, length(D_star))

    return D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star
end

function δ1(d_::Int, d::Int)
	return k1*ℯ^(-(d-d_)/τ1)
end

function δ2(d_::Int, d::Int)
	return k2*ℯ^(-(d-d_)/τ2)
end

function β(s::Int, p::Int, d::Int)
	return 1
end

function γ(t::Int, d::Int)
	return 1
end

function ρ(t::Int)
	return 2
end

file_path = input_files_folder * "/" * input_file_name
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

(56, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  47, 48, 49, 50, 51, 52, 53, 54, 55, 56], [6, 13, 20, 27, 34, 41, 48, 55], 18, SubString{String}["Player1", "Player2", "Player3", "Player4", "Player5", "Player6", "Player7", "Player8", "Player9", "Player10", "Player11", "Player12", "Player13", "Player14", "Player15", "Player16", "Player17", "Player18"], 38, SubString{String}["Ускорение", "Торможение", "Реакция", "ЛовляНаУрТуловища", "СменаХвата", "ПриставнойШаг", "СменаНаправления", "Лестница", "ПрофилактическиеУпражнения", "БеговыеУпражнения"  …  "Хаммер5Слабый", "Блейд5Сильный", "Хаммер10Сильный", "Блейд10Сильный", "Скубер10Сильный", "ФорхендБэкхенд25Вышагивание", "ФорхендБэкхенд50", "ХаммерБлейдСкубер25", "ПеревернутыйБэкхенд25", "ОбводнойБэкхенд25"], 10, SubString{String}["Зоны", "Роли", "ВертикальныйСтек", "ЛичнаяЗащита", "ГоризонтальныйСтек", "ДиагональныйСтек", "РазделенныйСтек", "СменаИПодстраховка", "АтакаВблизиГола", "СтандартныеПозиционныеСхемы"], 48, SubString{String}["Ускорение_1", "Т

In [30]:
# Модель
model = Model(SCIP.Optimizer)

A JuMP Model
├ solver: SCIP
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [31]:
# Containers for full solution
x_full = zeros(Float64, E, length(D_))
y_full = zeros(Float64, S, P, length(D_star))
z_full = zeros(Float64, T, P, length(D_star))
Z_full = zeros(Float64, T, length(D_star))

# Initial values, updated after each subtask
a0_cur = Float64.(a0)
b0_cur = [Float64.(b0[s]) for s in 1:S]
c0_cur = [Float64.(c0[t]) for t in 1:T]

for n in 1:length(D_star)
    start_day = (n == 1) ? 1 : D_star[n-1]
    end_day   = D_star[n]
    shift     = start_day - 1
    D_sub     = end_day - start_day + 1

    # Indices into global D_ for training days strictly after previous control date and up to current
    td_idx   = [i for i in 1:length(D_) if D_[i] > (start_day - 1) && D_[i] <= end_day]
    td_local = D_[td_idx] .- shift
    Ntd      = length(td_idx)

    h_sub    = [h[p][td_idx] for p in 1:P]
    V_sub    = V[td_idx]
    B_star_n = B_star[:, :, n]  # S × P
    C_star_n = C_star[:, :, n]  # T × P

    println("\n=== Subtask $n/$(length(D_star)): days $start_day..$end_day, $Ntd training sessions ===")

    sub_model = Model(SCIP.Optimizer)
    set_optimizer_attribute(sub_model, "limits/gap", 0.01)
    set_optimizer_attribute(sub_model, "limits/time", 1200.0)

    @variable(sub_model, x_s[1:E, 1:Ntd], Bin)
    @variable(sub_model, y_s[1:S, 1:P], Bin)
    @variable(sub_model, z_s[1:T, 1:P], Bin)
    @variable(sub_model, Z_s[1:T], Bin)
    @variable(sub_model, A_min_s)

    # Training duration constraints
    @constraint(sub_model, [d_=1:Ntd],
        sum(v[e]*x_s[e, d_] for e in 1:E) <= V_sub[d_])

    # Skill constraints: B(s,p) >= B_star_n[s,p] enforced via y_s
    @constraint(sub_model, [s=1:S, p=1:P],
        b0_cur[s][p]
        + sum(h_sub[p][d_] * sum(b[s][e]*x_s[e, d_] for e in 1:E) for d_ in 1:Ntd)
        + B_star_n[s, p] * (1 - y_s[s, p]) >= B_star_n[s, p])

    # Tactic constraints: C(t,p) >= C_star_n[t,p] enforced via z_s
    @constraint(sub_model, [t=1:T, p=1:P],
        c0_cur[t][p]
        + sum(h_sub[p][d_] * sum(c[t][e]*x_s[e, d_] for e in 1:E) for d_ in 1:Ntd)
        + C_star_n[t, p] * (1 - z_s[t, p]) >= C_star_n[t, p])

    # Tactic team constraints
    @constraint(sub_model, [t=1:T],
        sum(z_s[t, p] for p in 1:P) >= ρ(t) * Z_s[t])

    # A_min constraints (evaluated at control date D_sub in local coordinates)
    @constraint(sub_model, [p=1:P],
        A_min_s <= a0_cur[p]
            + sum(δ1(td_local[d_], D_sub) * h_sub[p][d_] * sum(w[e]*x_s[e, d_] for e in 1:E) for d_ in 1:Ntd)
            - sum(δ2(td_local[d_], D_sub) * h_sub[p][d_] * sum(w[e]*x_s[e, d_] for e in 1:E) for d_ in 1:Ntd))

    # Objective: equal 1/3 weights (no normalization)
    A_avg_obj = sum(
        a0_cur[p]
        + sum(δ1(td_local[d_], D_sub) * h_sub[p][d_] * sum(w[e]*x_s[e, d_] for e in 1:E) for d_ in 1:Ntd)
        - sum(δ2(td_local[d_], D_sub) * h_sub[p][d_] * sum(w[e]*x_s[e, d_] for e in 1:E) for d_ in 1:Ntd)
        for p in 1:P) / P

    A_obj = α * A_min_s + (1 - α) * A_avg_obj
    B_obj = sum(β(s, p, 1) * y_s[s, p] for s in 1:S, p in 1:P)
    C_obj = sum(γ(t, 1) * Z_s[t] for t in 1:T)

    @objective(sub_model, Max, (1/3)*A_obj + (1/3)*B_obj + (1/3)*C_obj)

    optimize!(sub_model)
    println("Status: $(termination_status(sub_model))")
    if termination_status(sub_model) == TIME_LIMIT
        println("Relative gap: $(relative_gap(sub_model)*100)%")
    end

    # Store results into full-problem containers
    for (local_d_, orig_d_) in enumerate(td_idx)
        for e in 1:E
            x_full[e, orig_d_] = value(x_s[e, local_d_])
        end
    end
    for s in 1:S, p in 1:P
        y_full[s, p, n] = value(y_s[s, p])
    end
    for t in 1:T, p in 1:P
        z_full[t, p, n] = value(z_s[t, p])
    end
    for t in 1:T
        Z_full[t, n] = value(Z_s[t])
    end

    # Update initial values for next subtask
    x_s_vals = value.(x_s)

    for p in 1:P
        a0_cur[p] += sum(
            (δ1(td_local[d_], D_sub) - δ2(td_local[d_], D_sub)) * h_sub[p][d_]
            * sum(w[e] * x_s_vals[e, d_] for e in 1:E)
            for d_ in 1:Ntd)
    end

    for s in 1:S, p in 1:P
        b0_cur[s][p] += sum(
            h_sub[p][d_] * sum(b[s][e] * x_s_vals[e, d_] for e in 1:E)
            for d_ in 1:Ntd)
    end

    for t in 1:T, p in 1:P
        c0_cur[t][p] += sum(
            h_sub[p][d_] * sum(c[t][e] * x_s_vals[e, d_] for e in 1:E)
            for d_ in 1:Ntd)
    end
end

println("\nAll subtasks solved.")



=== Subtask 1/8: days 1..6, 6 training sessions ===
feasible solution found by trivial heuristic after 0.0 seconds, objective value 1.625000e+01
presolving:
(round 1, fast)       0 del vars, 0 del conss, 0 add conss, 3 chg bounds, 864 chg sides, 864 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
   (0.0s) running MILP presolver
   (0.0s) MILP presolver (2 rounds): 1 aggregations, 0 fixings, 0 bound changes
(round 2, medium)     1 del vars, 898 del conss, 880 add conss, 5 chg bounds, 864 chg sides, 864 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 3, exhaustive) 1 del vars, 898 del conss, 880 add conss, 5 chg bounds, 864 chg sides, 864 chg coeffs, 870 upgd conss, 0 impls, 0 clqs, 0 implints
(round 4, medium)     1 del vars, 937 del conss, 1114 add conss, 5 chg bounds, 864 chg sides, 864 chg coeffs, 870 upgd conss, 0 impls, 234 clqs, 0 implints
(round 5, exhaustive) 1 del vars, 937 del conss, 1114 add conss, 5 chg bounds, 864 chg sides, 1367 chg coeffs, 870 upgd co

In [32]:
# Compute objective values from full solution
x_vals = round.(Int, x_full)
y_vals = round.(Int, y_full)
z_vals = round.(Int, z_full)
Z_vals = round.(Int, Z_full)

function compute_A_val(x_v)
    total = 0.0
    for d_ in 1:length(D_star)
        d_day = D_star[d_]
        A_min_val = minimum(
            a0[p] + sum((δ1(D_[di], d_day) - δ2(D_[di], d_day)) * h[p][di] * sum(w[e]*x_v[e, di] for e in 1:E)
                   for di in 1:length(D_) if D_[di] <= d_day)
            for p in 1:P)
        A_avg_val = sum(
            a0[p] + sum((δ1(D_[di], d_day) - δ2(D_[di], d_day)) * h[p][di] * sum(w[e]*x_v[e, di] for e in 1:E)
                   for di in 1:length(D_) if D_[di] <= d_day)
            for p in 1:P) / P
        total += α * A_min_val + (1 - α) * A_avg_val
    end
    return total
end

A_val = compute_A_val(x_vals)
B_val = Float64(sum(β(s, p, d_) * y_vals[s, p, d_] for s in 1:S, p in 1:P, d_ in 1:length(D_star)))
C_val = Float64(sum(γ(t, d_) * Z_vals[t, d_] for t in 1:T, d_ in 1:length(D_star)))
println("A = $A_val\nB = $B_val\nC = $C_val")

weights = [0.33, 0.33, 0.33]
output_folder = "results"
mkpath(output_folder)
output_file = "$(input_file_name)_$weights"

open(output_folder * "/" * output_file, "w") do io
    println(io, "Solution of model (subtasks) with weight distribution ", weights)

    println(io, "---Итоговая эффективность тренировок---")
    println(io, "A - общая физическая готовность команды\nB - бонус за освоенные индивидуальные навыки\nC - бонус за освоенные командой тактики\n")
    println(io, "A = $(A_val)\nB = $(B_val)\nC = $(C_val)\n\n")

    println(io, "---Расписание тренировок---\n")
    n_days = length(D_)
    data = Array{Any}(undef, n_days, 2)
    for d_ in 1:n_days
        data[d_, 1] = "Тренировка №$(d_) (день №$(D_[d_]))"
        day_exercises = String[]
        for e in 1:E
            if x_vals[e, d_] >= 1
                push!(day_exercises, Exercise_names[e])
            end
        end
        data[d_, 2] = join(day_exercises, ", ")
    end
    pretty_table(io, data, crop = :none)

    println(io, "---Освоенные навыки---\n")
    for p in 1:P
        for d_ in 1:length(D_star)
            println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
            learned_anything = false
            for s in 1:S
                if y_vals[s, p, d_] == 1 && (d_ == 1 || y_vals[s, p, d_-1] != 1)
                    learned_anything = true
                    println(io, "Освоил навык \'$(Skill_names[s])\'")
                end
            end
            if !learned_anything; println(io, "Ничего не освоил"); end
        end
        print(io, "\n")
    end

    println(io, "---Освоенные индивидуально тактики---\n")
    for p in 1:P
        for d_ in 1:length(D_star)
            println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
            learned_anything = false
            for t in 1:T
                if z_vals[t, p, d_] == 1 && (d_ == 1 || z_vals[t, p, d_-1] != 1)
                    learned_anything = true
                    println(io, "Освоил тактику \'$(Tactic_names[t])\'")
                end
            end
            if !learned_anything; println(io, "Ничего не освоил"); end
        end
        print(io, "\n")
    end

    println(io, "---Освоенные командой тактики---\n")
    for d_ in 1:length(D_star)
        for t in 1:T
            println(io, "Тактика: $(Tactic_names[t]), контрольная дата №$(d_) (день $(D_star[d_]))")
            println(io, Z_vals[t, d_] == 1 ? "Освоено" : "Не освоено")
        end
        print(io, "\n")
    end
end
println("Results written to: $(output_folder)/$(output_file)")


A = 1706.387886066867
B = 396.0
C = 0.0
Results written to: results/input_current.txt_[0.33, 0.33, 0.33]


In [33]:
# Write visualization data to txt file for Python visualizer
# Set ideal_point manually from scip_compute.ipynb if available:
ideal_point = [3654.73, 400.0, 20.0]

viz_file = output_folder * "/" * input_file_name * "_viz_data.txt"
open(viz_file, "w") do io
    println(io, "# Multi-criteria optimization visualization data")
    println(io, "# ideal_point,A,B,C")
    println(io, "# solution,wA,wB,wC,A,B,C,A_norm,B_norm,C_norm")
    if @isdefined(ideal_point)
        println(io, "ideal_point,$(ideal_point[1]),$(ideal_point[2]),$(ideal_point[3])")
        An_val = A_val / ideal_point[1]
        Bn_val = B_val / ideal_point[2]
        Cn_val = C_val / ideal_point[3]
    else
        println(io, "# ideal_point not defined - set it manually to enable normalization")
        An_val = NaN
        Bn_val = NaN
        Cn_val = NaN
    end
    wA, wB, wC = 0.33, 0.33, 0.33
    println(io, "solution,$wA,$wB,$wC,$A_val,$B_val,$C_val,$An_val,$Bn_val,$Cn_val")
end
println("Visualization data written to: $viz_file")


Visualization data written to: results/input_current.txt_viz_data.txt
